<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [8]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split


def load_data(seed):
    fashion_mnist = fetch_openml(
        "Fashion-MNIST",
        version=1,
        as_frame=False
    )

    X = fashion_mnist.data.astype("float32") / 255.0
    y = fashion_mnist.target.astype("int64")

    return train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )


X_train, X_val, y_train, y_val = load_data(seed=42)

print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)

Treino: (56000, 784) (56000,)
Validação: (14000, 784) (14000,)


Sim. Para uma MLP, a normalização é importante porque os pixels originalmente ficam na escala de 0 a 255. Ao trazer os valores para o intervalo entre 0 e 1, o treinamento fica mais estável, o gradiente tende a se comportar melhor e o modelo costuma convergir com menos dificuldade.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [9]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=30,
        batch_size=128,
        random_state=seed
    )

    print("Treinando MLP...")
    print(f"Ativação: {activation}")
    print(f"Camadas escondidas: {hidden_layers}")
    print(f"Learning rate: {learning_rate}")

    model.fit(X_train, y_train)

    print("MLP treinada com sucesso")
    print(f"Épocas executadas: {model.n_iter_}")

    return model


mlp_model = train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=42
)

print("Modelo criado:", type(mlp_model).__name__)

Treinando MLP...
Ativação: relu
Camadas escondidas: (64,)
Learning rate: 0.001
MLP treinada com sucesso
Épocas executadas: 30
Modelo criado: MLPClassifier


c:\Users\nunes\github-classroom\ddefbcourses\atividade-03-mlp-Jvnunes615\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [10]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

    print("Avaliação do modelo")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1-score: {metrics['f1_score']:.4f}")

    return metrics


validation_metrics = evaluate(mlp_model, X_val, y_val)

Avaliação do modelo
Accuracy: 0.8825
Precision: 0.8834
Recall: 0.8825
F1-score: 0.8823


A função `evaluate` usa o modelo treinado para prever as classes do conjunto de validação e calcula as principais métricas de classificação. Como o Fashion MNIST possui várias classes, precision, recall e f1-score foram calculados com média ponderada para considerar o tamanho de cada classe.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [14]:
import logging
import time
import warnings

import mlflow
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier

logging.disable(logging.WARNING)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
mlflow.set_experiment("assignment")

params = {
    "activation": "relu",
    "hidden_layers": (64,),
    "learning_rate": 0.001,
    "max_iter": 30,
    "batch_size": 128
}

if mlflow.active_run() is not None:
    mlflow.end_run()

with mlflow.start_run(run_name="mlp_baseline"):
    start_time = time.time()

    model = MLPClassifier(
        hidden_layer_sizes=params["hidden_layers"],
        activation=params["activation"],
        solver="adam",
        learning_rate_init=params["learning_rate"],
        max_iter=params["max_iter"],
        batch_size=params["batch_size"],
        random_state=42
    )

    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    y_pred = model.predict(X_val)
    metrics = {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
        "training_time": training_time
    }

    mlflow.log_params({
        "activation": params["activation"],
        "hidden_layers": str(params["hidden_layers"]),
        "learning_rate": params["learning_rate"],
        "max_iter": params["max_iter"],
        "batch_size": params["batch_size"]
    })
    mlflow.log_metrics(metrics)

logging.disable(logging.NOTSET)

print(f"activation: {params['activation']}")
print(f"hidden_layers: {params['hidden_layers']}")
print(f"learning_rate: {params['learning_rate']}")
print(f"max_iter: {params['max_iter']}")
print(f"batch_size: {params['batch_size']}")
print(f"accuracy: {metrics['accuracy']:.4f}")
print(f"precision: {metrics['precision']:.4f}")
print(f"recall: {metrics['recall']:.4f}")
print(f"f1_score: {metrics['f1_score']:.4f}")
print(f"training_time: {metrics['training_time']:.2f}s")

activation: relu
hidden_layers: (64,)
learning_rate: 0.001
max_iter: 30
batch_size: 128
accuracy: 0.8825
precision: 0.8834
recall: 0.8825
f1_score: 0.8823
training_time: 32.60s


# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [15]:
import logging
import time
import warnings

import mlflow
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
mlflow.set_experiment("assignment")

base_params = {
    "hidden_layers": (64,),
    "learning_rate": 0.001,
    "max_iter": 30,
    "batch_size": 128,
    "seed": 42
}


def run_activation_experiment(activation):
    if mlflow.active_run() is not None:
        mlflow.end_run()

    with mlflow.start_run(run_name=f"activation_{activation}"):
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=base_params["hidden_layers"],
            activation=activation,
            solver="adam",
            learning_rate_init=base_params["learning_rate"],
            max_iter=base_params["max_iter"],
            batch_size=base_params["batch_size"],
            random_state=base_params["seed"]
        )

        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        y_pred = model.predict(X_val)

        metrics = {
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
            "training_time": training_time,
            "loss": model.loss_,
            "epochs": model.n_iter_
        }

        mlflow.log_params({
            "activation": activation,
            "hidden_layers": str(base_params["hidden_layers"]),
            "learning_rate": base_params["learning_rate"],
            "max_iter": base_params["max_iter"],
            "batch_size": base_params["batch_size"]
        })
        mlflow.log_metrics(metrics)

    return {"activation": activation, **metrics}


activations = ["logistic", "tanh", "relu"]
activation_results = [run_activation_experiment(name) for name in activations]
activation_results_df = pd.DataFrame(activation_results)

columns = ["activation", "accuracy", "precision", "recall", "f1_score", "loss", "epochs", "training_time"]
print(activation_results_df[columns].to_string(index=False, float_format="{:.4f}".format))

best_activation = activation_results_df.loc[activation_results_df["f1_score"].idxmax(), "activation"]
print(f"\nmelhor_ativacao: {best_activation}")

activation  accuracy  precision  recall  f1_score   loss  epochs  training_time
  logistic    0.8856     0.8858  0.8856    0.8854 0.2359      30        29.1352
      tanh    0.8870     0.8872  0.8870    0.8867 0.1867      30        22.2553
      relu    0.8825     0.8834  0.8825    0.8823 0.2105      30        35.7005

melhor_ativacao: tanh


## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

**Resposta:**

A melhor convergência deve ser analisada pela combinação entre `f1_score`, `loss` e `epochs`. No geral, a ativação com maior `f1_score` e menor `loss` é a melhor candidata.

A maior estabilidade aparece quando a ativação mantém bom desempenho sem perda muito alta e sem tempo de treinamento exagerado. Em MLPs para imagens normalizadas, `relu` costuma ser a opção mais eficiente, enquanto `logistic` tende a convergir mais lentamente.

Sim, podem existir diferenças significativas de treinamento. As ativações mudam o comportamento dos gradientes, afetando velocidade de convergência, perda final e desempenho no conjunto de validação.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [16]:
import logging
import time
import warnings

import mlflow
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
mlflow.set_experiment("assignment")

architecture_params = {
    "activation": "relu",
    "learning_rate": 0.001,
    "max_iter": 30,
    "batch_size": 128,
    "seed": 42
}


def run_architecture_experiment(hidden_layers):
    if mlflow.active_run() is not None:
        mlflow.end_run()

    architecture_name = str(hidden_layers)

    with mlflow.start_run(run_name=f"architecture_{architecture_name}"):
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=architecture_params["activation"],
            solver="adam",
            learning_rate_init=architecture_params["learning_rate"],
            max_iter=architecture_params["max_iter"],
            batch_size=architecture_params["batch_size"],
            random_state=architecture_params["seed"]
        )

        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        y_pred = model.predict(X_val)

        metrics = {
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
            "training_time": training_time,
            "loss": model.loss_,
            "epochs": model.n_iter_
        }

        mlflow.log_params({
            "activation": architecture_params["activation"],
            "hidden_layers": architecture_name,
            "learning_rate": architecture_params["learning_rate"],
            "max_iter": architecture_params["max_iter"],
            "batch_size": architecture_params["batch_size"]
        })
        mlflow.log_metrics(metrics)

    return {"hidden_layers": architecture_name, **metrics}


architectures = [(32,), (64,), (128, 64), (256, 128)]
architecture_results = [run_architecture_experiment(layers) for layers in architectures]
architecture_results_df = pd.DataFrame(architecture_results)

columns = ["hidden_layers", "accuracy", "precision", "recall", "f1_score", "loss", "epochs", "training_time"]
print(architecture_results_df[columns].to_string(index=False, float_format="{:.4f}".format))

best_architecture = architecture_results_df.loc[architecture_results_df["f1_score"].idxmax(), "hidden_layers"]
print(f"\nmelhor_arquitetura: {best_architecture}")

hidden_layers  accuracy  precision  recall  f1_score   loss  epochs  training_time
        (32,)    0.8790     0.8804  0.8790    0.8795 0.2636      30        26.2613
        (64,)    0.8825     0.8834  0.8825    0.8823 0.2105      30        34.5639
    (128, 64)    0.8907     0.8914  0.8907    0.8904 0.1480      30        64.4133
   (256, 128)    0.8983     0.8990  0.8983    0.8983 0.1189      30       133.8620

melhor_arquitetura: (256, 128)


## Responda:

- Redes maiores sempre melhoraram os resultados?
- Qual foi o impacto no tempo de treinamento?
- Qual arquitetura apresentou melhor tradeoff?

**Resposta:**

Redes maiores não necessariamente melhoram os resultados. Elas aumentam a capacidade do modelo, mas também podem elevar o tempo de treinamento e o risco de ajuste excessivo aos dados de treino.

A arquitetura com melhor tradeoff deve equilibrar bom `f1_score`, baixa `loss` e tempo de treinamento razoável. Por isso, a melhor escolha não é apenas a maior rede, mas aquela que entrega desempenho parecido ou superior sem custo computacional desnecessário.

Depois de rodar a célula, a coluna `melhor_arquitetura` indica a melhor opção pelo `f1_score`, enquanto a tabela ajuda a comparar se o ganho compensa o tempo de treinamento.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [18]:
import logging
import time
import warnings

import mlflow
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier

logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
mlflow.set_experiment("assignment")

learning_rate_params = {
    "activation": "relu",
    "hidden_layers": (64,),
    "max_iter": 30,
    "batch_size": 128,
    "seed": 42
}


def run_learning_rate_experiment(learning_rate):
    if mlflow.active_run() is not None:
        mlflow.end_run()

    with mlflow.start_run(run_name=f"learning_rate_{learning_rate}"):
        start_time = time.time()

        model = MLPClassifier(
            hidden_layer_sizes=learning_rate_params["hidden_layers"],
            activation=learning_rate_params["activation"],
            solver="adam",
            learning_rate_init=learning_rate,
            max_iter=learning_rate_params["max_iter"],
            batch_size=learning_rate_params["batch_size"],
            random_state=learning_rate_params["seed"]
        )

        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        y_pred = model.predict(X_val)

        metrics = {
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_val, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y_val, y_pred, average="weighted", zero_division=0),
            "training_time": training_time,
            "loss": model.loss_,
            "epochs": model.n_iter_
        }

        mlflow.log_params({
            "activation": learning_rate_params["activation"],
            "hidden_layers": str(learning_rate_params["hidden_layers"]),
            "learning_rate": learning_rate,
            "max_iter": learning_rate_params["max_iter"],
            "batch_size": learning_rate_params["batch_size"]
        })
        mlflow.log_metrics(metrics)

    return {"learning_rate": learning_rate, **metrics}


learning_rates = [0.1, 0.01, 0.001]
learning_rate_results = [run_learning_rate_experiment(rate) for rate in learning_rates]
learning_rate_results_df = pd.DataFrame(learning_rate_results)

columns = ["learning_rate", "accuracy", "precision", "recall", "f1_score", "loss", "epochs", "training_time"]
print(learning_rate_results_df[columns].to_string(index=False, float_format="{:.4f}".format))

best_learning_rate = learning_rate_results_df.loc[learning_rate_results_df["f1_score"].idxmax(), "learning_rate"]
print(f"\nmelhor_learning_rate: {best_learning_rate}")

 learning_rate  accuracy  precision  recall  f1_score   loss  epochs  training_time
        0.1000    0.5088     0.4989  0.5088    0.4572 1.0903      13        24.2592
        0.0100    0.8700     0.8733  0.8700    0.8707 0.2749      30        66.0876
        0.0010    0.8825     0.8834  0.8825    0.8823 0.2105      30        33.7308

melhor_learning_rate: 0.001


## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

**Resposta:**

Learning rates maiores podem deixar o treinamento instável, porque os pesos são atualizados com passos grandes demais e o modelo pode ter dificuldade para reduzir a perda de forma consistente.

Learning rates muito baixos tendem a ser mais estáveis, mas podem convergir lentamente e precisar de mais épocas para chegar a um bom resultado.

O melhor comportamento deve ser escolhido pela combinação entre maior `f1_score`, menor `loss` e tempo de treinamento aceitável. Depois de rodar a célula, `melhor_learning_rate` mostra a melhor opção pelo `f1_score`, e a tabela ajuda a verificar se ela também foi estável.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


**Resposta:**

A ativação com melhor desempenho foi `tanh`, pois apresentou o maior `f1_score` entre as funções testadas. A diferença para `logistic` foi pequena, mas `tanh` teve desempenho ligeiramente superior, enquanto `relu` ficou um pouco abaixo nessa comparação.

A arquitetura com melhor resultado bruto foi `(256, 128)`, com o maior `f1_score`. Porém, considerando tradeoff, a arquitetura `(128, 64)` foi uma escolha mais equilibrada: teve desempenho próximo ao da maior rede, mas com tempo de treinamento bem menor.

O learning rate mais estável foi `0.001`. O valor `0.1` mostrou comportamento ruim e instável, com queda forte nas métricas, indicando que passos muito grandes prejudicaram o treinamento.

Não há evidência forte de overfitting apenas pelos resultados salvos, porque a análise foi feita principalmente no conjunto de validação. Ainda assim, redes maiores, como `(256, 128)`, têm maior risco de overfitting e deveriam ser acompanhadas comparando desempenho de treino e validação.

A melhor configuração final entre os experimentos realizados foi a rede `(256, 128)` com `relu` e `learning_rate=0.001`, pois obteve o maior desempenho de validação. Se o objetivo for equilíbrio entre desempenho e custo, eu escolheria `(128, 64)` com `learning_rate=0.001`.

As principais dificuldades observadas foram o tempo de treinamento, a sensibilidade aos hiperparâmetros e a necessidade de comparar os modelos por várias métricas, não apenas por accuracy.